In [ ]:
%%capture
import os
from pathlib import Path

import numpy as np
import pandas as pd
from datetime import datetime
from dj_notebook import activate

env_file = os.environ["META_ENV"]
reports_folder = Path(os.environ["META_REPORTS_FOLDER"])
analysis_folder = Path(os.environ["META_ANALYSIS_FOLDER"])
pharmacy_folder = Path(os.environ["META_PHARMACY_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option("future.no_silent_downcasting", True)

In [ ]:
# export from patient history
# focus on ARV regimen -- specifically flag DTG regimens
# for katie 14 AUG 2026

In [ ]:
from edc_pdutils.dataframes import get_crf
from clinicedc_constants import NO, YES


In [ ]:
df_patient_history = get_crf(
    "meta_subject.patienthistory", subject_visit_model="meta_subject.subjectvisit"
)

In [ ]:
for col in ["other_oi_prophylaxis", "subject_identifier", "subject_visit_id"]:
    df_patient_history[col] = df_patient_history[col].astype("string")
    df_patient_history[col] = df_patient_history[col].str.strip().replace("", pd.NA)

df_patient_history["dob"] = pd.to_datetime(df_patient_history["dob"])

In [ ]:
df_patient_history["dob"] = pd.to_datetime(df_patient_history["dob"])

for column in df_patient_history.select_dtypes(include="string").columns:
    df_patient_history[column] = df_patient_history[column].astype("string")
    df_patient_history[column] = df_patient_history[column].str.strip().replace("", pd.NA)

for column in df_patient_history.select_dtypes(include="datetimetz").columns:
    df_patient_history[column] = df_patient_history[column].dt.tz_convert("utc").dt.normalize()


In [ ]:
df_patient_history[["current_arv_regimen", "other_current_arv_regimen"]]
df_patient_history["current_arv_regimen"] = np.where(df_patient_history["current_arv_regimen"]=="Other, specify ...", df_patient_history["other_current_arv_regimen"], df_patient_history["current_arv_regimen"])


In [ ]:
df_patient_history["current_arv_regimen"] = df_patient_history["current_arv_regimen"].replace("TDF+FTC+DTG", "TDF + FTC + DTG")
df_patient_history["current_arv_regimen"] = df_patient_history["current_arv_regimen"].replace("TDF+3TC+DTG", "TDF + 3TC + DTG")
df_patient_history["current_arv_regimen"] = df_patient_history["current_arv_regimen"].replace("ABC+3TC+DTG", "ABC + 3TC + DTG")
df_patient_history["current_arv_regimen"] = df_patient_history["current_arv_regimen"].replace("ABC+ 3TC+ DTG", "ABC + 3TC + DTG")
df_patient_history["current_arv_regimen"] = df_patient_history["current_arv_regimen"].replace("AZT+3TC+DTG", "AZT + 3TC + DTG")

df_patient_history["dtg"] = np.where(df_patient_history["current_arv_regimen"].str.contains("DTG"), YES, NO)



In [ ]:
df_export = df_patient_history[["subject_identifier", "gender", "dob", "baseline_datetime", "visit_datetime", "visit_code", "current_arv_regimen", "dtg", "current_arv_regimen_start_date", "arv_initiation_date"]].sort_values(by="subject_identifier").reset_index(drop=True)

for column in df_export.select_dtypes(include="datetimetz").columns:
    df_export[column] = df_export[column].dt.tz_localize(None)


In [ ]:
df_export.dtypes

In [ ]:
tstamp = datetime.today().strftime('%Y%m%d%H%M')

df_export.to_stata(analysis_folder / f"patient_history_dtg_{tstamp}.dta",version=118,write_index=False)

In [ ]:
df_export

In [ ]:
df_patient_history[["subject_identifier", "current_arv_regimen"]].groupby(
    by=["current_arv_regimen"]
)["current_arv_regimen"].value_counts()

In [ ]:
df_patient_history[["subject_identifier", "current_arv_regimen", "other_current_arv_regimen"]][
    ["other_current_arv_regimen"]
].value_counts()

In [ ]:
df_patient_history["other_current_arv_regimen"] = df_patient_history[
    "other_current_arv_regimen"
].apply(lambda x: x.split("-")[0])